# Developing an agent

These examples use this checkout of New RecSim. From the repository root, create a virtual environment and run `python -m pip install -e ".[rl,docs]"`. Select that Python environment as your notebook kernel. All examples run on CPU and write only to temporary directories. The simulator itself does not require PyTorch.

Copyright 2019 The RecSim Authors. Licensed under the Apache License, Version 2.0.

## Lifecycle and action semantics

An agent implements `step(reward, observation)` and may override `begin_episode` and `end_episode`. Actions index the candidate slots. Keep policy randomness in the agent; do not seed global NumPy. Learning agents consume the final transition and distinguish natural termination from truncation.

In [ ]:
import numpy as np
from recsim.agent import AbstractEpisodicRecommenderAgent
from recsim.environments import interest_evolution


class HighestAffinityAgent(AbstractEpisodicRecommenderAgent):
    def step(self, reward, observation):
        features = np.stack(list(observation["doc"].values()))
        scores = features @ observation["user"]
        return np.argsort(-scores, kind="stable")[: self._slate_size].tolist()


env = interest_evolution.create_environment(dict(num_candidates=6, slate_size=2, seed=3))
obs, _ = env.reset(seed=3)
agent = HighestAffinityAgent(env.action_space)
action = agent.begin_episode(obs)
assert env.action_space.contains(np.asarray(action))
print("Chosen slots:", action)
obs, reward, terminated, truncated, _ = env.step(action)
agent.end_episode(reward, obs, terminated=terminated, truncated=True)
env.close()

## A known MDP as a learning reference

The included six-state test environment has known Q-values. A tabular agent with gamma=0 should learn immediate rewards. This checks numerical behavior before moving to a neural approximator.

In [ ]:
from recsim.testing import test_environment
from recsim.agents.tabular_q_agent import TabularQAgent

env = test_environment.create_environment(
    dict(
        num_candidates=4,
        slate_size=1,
        seed=42,
        resample_documents=False,
        starting_probs=(1.0, 0.0, 0.0, 0.0, 0.0, 0.0),
    )
)
agent = TabularQAgent(env.observation_space, env.action_space, gamma=0.0, learning_rate=0.8)
obs, _ = env.reset(seed=42)
action = agent.begin_episode(obs)
for step in range(2500):
    obs, reward, terminated, truncated, _ = env.step(action)
    if step == 2499:
        agent.end_episode(reward, obs, terminated=terminated, truncated=True)
    else:
        action = agent.step(reward, obs)
learned = np.array([[agent._q_value_table[(a, s)] for a in range(4)] for s in range(6)])
np.testing.assert_allclose(learned, test_environment.QVALUES0, atol=1e-4)
env.close()
print(learned)

## State and reproducibility

New runners use `state_dict()` and `load_state_dict()`. These include counters, tables, local RNG state and layer children. Torch agents additionally serialize online/target networks, optimizer and replay. The old bundle methods remain compatibility entry points, but the base legacy bundle is intentionally minimal.

In [ ]:
state = agent.state_dict()
clone = TabularQAgent(env.observation_space, env.action_space, gamma=0.0, learning_rate=0.8)
clone.load_state_dict(state)
assert clone._q_value_table == agent._q_value_table
print("Restored state-action entries:", len(clone._q_value_table))

## Hierarchical agents

Layers transform observations and rewards before delegating to child agents. Click statistics start with zero impressions. Fixed-length history uses valid padding rather than `None`. Temporal aggregation needs matching document features across held actions, so use a fixed candidate pool or a suitable document comparator.

In [ ]:
from recsim.environments import interest_exploration
from recsim.agents.layers.cluster_click_statistics import ClusterClickStatsLayer
from recsim.agents.cluster_bandit_agent import ClusterBanditAgent

env = interest_exploration.create_environment(dict(num_candidates=10, slate_size=2, seed=5))
layer = ClusterClickStatsLayer(ClusterBanditAgent, env.observation_space, env.action_space)
obs, _ = env.reset(seed=5)
action = layer.begin_episode(obs)
assert sum(layer._sufficient_statistics["impression_count"]) == 0
for step in range(5):
    obs, reward, terminated, truncated, _ = env.step(action)
    if step == 4:
        layer.end_episode(reward, obs, terminated=False, truncated=True)
    else:
        action = layer.step(reward, obs)
env.close()
print("Layer rollout completed")